In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import httpx
from langchain_groq import ChatGroq

# Disable SSL verification to work around corporate proxy/firewall
client = httpx.Client(verify=False)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, http_client=client)

In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [4]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[sql_query]
)

In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [7]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='f59db2b2-9cb9-435f-976a-4a0c92e5d89e'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '6369gbztt', 'function': {'arguments': '{"query":"SELECT name FROM artists WHERE name LIKE \'S%\' ORDER BY popularity DESC LIMIT 1"}', 'name': 'sql_query'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 233, 'total_tokens': 264, 'completion_time': 0.077876487, 'completion_tokens_details': None, 'prompt_time': 0.026103, 'prompt_tokens_details': None, 'queue_time': 0.04914288, 'total_time': 0.103979487}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc42b-693e-73c0-b4a1-1cedd0c72c1c-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELE

In [8]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT ArtistId, Name FROM Artist WHERE Name LIKE 'S%' ORDER BY Name LIMIT 1
